# What does Seeker look at?

<img src="multimedia/sim_1.gif" alt="Seeker tracking task-relevant regions through a robot demonstration" width="900">

Seeker receives two camera observations together with task and robot context. For each frame, it predicts a **soft spatial mask** and a corresponding **region of interest (ROI)** that can be used as a visual bottleneck by a robot policy.

This notebook runs the released checkpoint over one recorded MimicGen demonstration. No training is required.

> **Prerequisites:** activate the `seeker` environment, run `seeker setup`, and rerender at least one task as described in the [README quickstart](README.md#quickstart).

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Video, display

if not Path("setup.py").is_file():
    raise RuntimeError("Launch Jupyter from the Seeker repository root.")

from seeker.dataset.cache import resolve_cache_dir
from seeker.dataset.mimicgen_dataset import MimicGenDataset
from seeker.model.base_encoder import BaseEncoder
from seeker.model.seeker import Seeker
from seeker.util.image_ops import image_to_float01
from seeker.util.roi import grid_mask_to_pixel_box
from seeker.util.visualization import (
    overlay_boxes_on_images,
    overlay_masks_on_images,
    save_attention_heads_video,
    visualize_trajectory,
)

# Edit these values to inspect another cached demonstration.
DATASET_PATH = Path("datasets/mimicgen/three_piece_assembly_d2/three_piece_assembly_d2.hdf5")
EPISODE_IDX = 0
BATCH_SIZE = 16 if torch.cuda.is_available() else 4
SHOW_HEAD_DETAILS = False

WEIGHTS_PATH = Path(".weights/seeker.mimicgen.pth")
DINO_WEIGHTS_PATH = Path(".weights/dinov3.vits16plus.pth")
OUTPUT_DIR = Path("outputs/seeker_demo")
VIT_IN = 224
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CACHE_DIR = Path(resolve_cache_dir(str(DATASET_PATH)))

required_paths = {
    "dataset": DATASET_PATH,
    "rerendered cache": CACHE_DIR,
    "Seeker checkpoint": WEIGHTS_PATH,
    "DINOv3 checkpoint": DINO_WEIGHTS_PATH,
}
missing = [f"{name}: {path}" for name, path in required_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing demo prerequisites:\n- "
        + "\n- ".join(missing)
        + "\nRun `seeker setup` for weights and the README rerender command for data."
    )

SHAPE_META = {
    "obs": {
        "agentview_image": {"shape": [3, 256, 256], "type": "rgb"},
        "robot0_eye_in_hand_image": {"shape": [3, 256, 256], "type": "rgb"},
        "robot0_eef_pos": {"shape": [3]},
        "robot0_eef_rot": {"shape": [9]},
        "robot0_gripper_qpos": {"shape": [2]},
    },
    "action": {"shape": [10]},
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE} | cache: {CACHE_DIR}")

## What Seeker receives

The checkpoint sees a third-person **agentview** and a wrist-mounted **eye-in-hand** view. It also receives the task embedding, end-effector pose, gripper state, and robot identity.

In [ ]:
dataset = MimicGenDataset(
    shape_meta=SHAPE_META,
    dataset_path=str(DATASET_PATH),
    image_size=VIT_IN,
    horizon=1,
    val_ratio=0.0,
    cache_dir=str(CACHE_DIR),
)

trajectory = dataset.get_trajectory(EPISODE_IDX)
episode_length = len(trajectory["obs"]["agentview_image"])
frame_indices = np.arange(episode_length, dtype=np.int64)

# BaseEncoder expects a temporal dimension, so each frame becomes [frame, T=1, ...].
obs = {
    key: torch.from_numpy(np.ascontiguousarray(value[frame_indices])).unsqueeze(1)
    for key, value in trajectory["obs"].items()
}
T = len(frame_indices)

preview_positions = np.linspace(0, T - 1, num=min(4, T), dtype=np.int64)
fig, axes = plt.subplots(2, len(preview_positions), figsize=(3 * len(preview_positions), 6), squeeze=False)
camera_rows = [
    ("agentview_image", "Agentview"),
    ("robot0_eye_in_hand_image", "Eye in hand"),
]
for row, (key, label) in enumerate(camera_rows):
    frames = trajectory["obs"][key][frame_indices]
    for col, position in enumerate(preview_positions):
        axes[row, col].imshow(np.moveaxis(frames[position], 0, -1))
        axes[row, col].set_title(f"t={frame_indices[position]}")
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(label, fontsize=12)
plt.suptitle(f"Recorded demonstration {EPISODE_IDX}: {T} frames")
plt.tight_layout()
plt.show()

## From observations to visual focus

```text
camera views + task + robot state
                  │
                  ▼
        Seeker coarse stage
                  │
                  ▼
       soft attention mask + ROI
                  │
                  ▼
            policy input
```

This is the inference-time path: every frame is evaluated independently with the **coarse** stage. The released checkpoint can predict focus for both views, so both are shown here. In the repository's default `method=seeker` policy profile, agentview is cropped using this prediction while eye-in-hand is passed through unchanged.

In [ ]:
seeker = Seeker(
    weights=str(WEIGHTS_PATH),
    views=["agentview", "eye_in_hand"],
    verbose=False,
    strict_weights=True,
).eval()

encoder = BaseEncoder()
encoder.enable_eih = True
enc_in = encoder.obs_to_input(obs, seeker.normalizer, resize=False)
seeker = seeker.to(DEVICE)

def infer_coarse(images, view):
    collected = {"mask": [], "attn_map": [], "head_score": []}
    with torch.inference_mode():
        for start in range(0, images.shape[0], BATCH_SIZE):
            stop = min(start + BATCH_SIZE, images.shape[0])
            composer_batch = {
                key: value[start:stop].to(DEVICE)
                for key, value in enc_in.composer_in.items()
            }
            coarse = seeker(
                image=images[start:stop].to(DEVICE),
                view=view,
                composer_in=composer_batch,
                stage="coarse",
            ).coarse
            for name in collected:
                collected[name].append(getattr(coarse, name).detach().cpu())
    return {name: torch.cat(parts, dim=0) for name, parts in collected.items()}

started = time.perf_counter()
agent_prediction = infer_coarse(enc_in.agentview, "agentview")
eih_prediction = infer_coarse(enc_in.eye_in_hand, "eye_in_hand")
elapsed = time.perf_counter() - started

image_size = enc_in.agentview.shape[-1]
full_box = torch.tensor(
    [[0.0, 0.0, image_size - 1.0, image_size - 1.0]]
).expand(T, -1)
agent_box = grid_mask_to_pixel_box(agent_prediction["mask"].squeeze(1), full_box)
eih_box = grid_mask_to_pixel_box(eih_prediction["mask"].squeeze(1), full_box)

print(f"Processed {T} frames from both views on {DEVICE} in {elapsed:.1f}s.")

## What Seeker produces

For each camera, Seeker produces a soft mask over image patches. The selected-content panel shows that mask at image resolution; the ROI is the tight square region derived from it.

In [ ]:
keyframe = T // 2
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
rows = [
    ("Agentview", enc_in.agentview, agent_prediction, agent_box),
    ("Eye in hand", enc_in.eye_in_hand, eih_prediction, eih_box),
]
for row, (label, images, prediction, boxes) in enumerate(rows):
    image = image_to_float01(images[keyframe:keyframe + 1], source="imagenet")
    selected = overlay_masks_on_images(
        image, prediction["mask"][keyframe:keyframe + 1], blackout=True
    )
    roi = overlay_boxes_on_images(image, boxes[keyframe:keyframe + 1])
    for col, panel in enumerate((image, selected, roi)):
        axes[row, col].imshow(panel[0].permute(1, 2, 0).numpy())
        axes[row, col].axis("off")
    axes[row, 0].set_ylabel(label, fontsize=12)

for col, title in enumerate(("Input", "Mask-selected content", "Predicted ROI")):
    axes[0, col].set_title(title)
plt.suptitle(f"One coarse prediction at recorded timestep {frame_indices[keyframe]}")
plt.tight_layout()
plt.show()

## Follow Seeker's focus through the demonstration

Each row is one camera. The left panel shows the predicted ROI; the right panel retains the content selected by the soft mask. The gripper bar supplies temporal context about the recorded manipulation.

In [ ]:
visualize_trajectory(
    images=enc_in.agentview,
    mask=agent_prediction["mask"],
    boxes=[agent_box],
    eih_images=enc_in.eye_in_hand,
    eih_mask=eih_prediction["mask"],
    eih_boxes=[eih_box],
    gripper_opening=enc_in.composer_in["gripper_opening"],
    blackout=True,
    save_dir=str(OUTPUT_DIR),
    prefix="seeker_focus",
    save_video=True,
)
display(Video(str(OUTPUT_DIR / "seeker_focus.mp4"), embed=True))

## Optional: how attention heads coordinate

The coarse stage contains six attention heads. Each proposes a spatial distribution, while its coordination score controls its contribution to the combined mask. Set `SHOW_HEAD_DETAILS = True` in the setup cell to render these diagnostics.

In [ ]:
if not SHOW_HEAD_DETAILS:
    print("Optional head details are disabled. Set SHOW_HEAD_DETAILS=True to render them.")
else:
    head_videos = []
    for view, images, prediction in (
        ("agentview", enc_in.agentview, agent_prediction),
        ("eye_in_hand", enc_in.eye_in_hand, eih_prediction),
    ):
        path = OUTPUT_DIR / f"heads_{view}.mp4"
        save_attention_heads_video(
            images,
            prediction["attn_map"],
            prediction["head_score"],
            str(path),
            save_frames_every=None,
        )
        head_videos.append(path)
    for path in head_videos:
        display(Video(str(path), embed=True))

## Takeaways

- Seeker conditions visual focus on camera observations, task identity, and robot state.
- Its inference-time output is a coarse soft mask and a corresponding ROI.
- A downstream policy decides how to consume that focus; the default profile crops agentview and passes eye-in-hand through.

Continue with the [training guide](README.md#training), read the [checkpoint provenance](seeker/model/WEIGHTS.md), or see the [paper](multimedia/paper.pdf).